# Файнтюн Gemma 4 E2B под пайплайн FuckHR

Одна модель на все 11 этапов: `extract`, `hr_filter`, `company`, `contacts`, `dossier`,
`review_fake`, `ai_text`, `draft`, `intake`, `resume_section`, `resume_tailor`.
Несколько LoRA-адаптеров не берём: в Ollama каждый адаптер — отдельная модель, а при
`OLLAMA_MAX_LOADED_MODELS=1` переключение этапов означало бы перезагрузку весов.

Почему E2B, а не E4B: 8 ГБ VRAM и `bge-m3` рядом. E2B в Q4 ≈ 2.9 ГБ, эмбеддер ≈ 1.3 ГБ —
обе модели живут в памяти одновременно, свопов между этапами нет. Unsloth рекомендует
E4B, но у него Q4 ≈ 5 ГБ, и эмбеддер начнёт вытесняться.

Обучаем в non-thinking режиме (`chat_template = "gemma-4"`): датасет собран из финальных
ответов, а рассуждения на 8 ГБ стоят секунд на каждом вызове.

**Runtime → Change runtime type → T4 GPU.** Времени на 738 примеров — примерно 20–35 минут.


## 1. Установка


In [ ]:
%%capture
# Только unsloth: версии transformers/tokenizers/hub сводит pip.
# Ручные пины здесь ломают окружение — в образах Colab они разные.
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


Проверка окружения до импорта `unsloth`. Она не знает «правильных» версий заранее:
берёт требования установленного `transformers` и сверяет с тем, что стоит в образе.
Если версии разошлись — выполните напечатанную команду и сделайте
**Runtime → Restart session**: пакеты уже загружены в память, без рестарта
апгрейд не подхватится.


In [ ]:
# Сверяем окружение с тем, что объявил сам установленный transformers.
# Свои числа не проставляем: в разных образах Colab границы разные.
from importlib.metadata import requires, version

from packaging.requirements import Requirement
from packaging.version import Version

WATCH = {'tokenizers', 'huggingface-hub', 'safetensors', 'accelerate'}
broken = []
for raw in requires('transformers') or []:
    req = Requirement(raw)
    if req.name not in WATCH:
        continue
    if req.marker is not None and not req.marker.evaluate():
        continue
    try:
        got = Version(version(req.name))
    except Exception:
        continue
    if req.specifier and got not in req.specifier:
        broken.append((req.name, str(req.specifier), str(got)))

print('transformers', version('transformers'))
for name in sorted(WATCH):
    try:
        print(' ', name, version(name))
    except Exception:
        pass

if broken:
    fix = ' '.join('"{}{}"'.format(n, s) for n, s, _ in broken)
    raise SystemExit(
        'Версии разошлись: '
        + '; '.join('{} нужен {}, стоит {}'.format(n, s, g) for n, s, g in broken)
        + '\n\n  !pip install --upgrade --no-cache-dir ' + fix
        + '\n\nЗатем Runtime -> Restart session и продолжайте со следующей ячейки.'
    )

import unsloth
print('unsloth', version('unsloth'), 'готов')


## 2. Датасет

`train.jsonl` и `val.jsonl` собирает `training/make_dataset.py` из репозитория.
Загрузите оба файла кнопкой ниже (или подмонтируйте Google Drive и поправьте пути).


In [ ]:
import json, os
from pathlib import Path

TRAIN, VAL = Path('train.jsonl'), Path('val.jsonl')
if not TRAIN.exists():
    from google.colab import files
    files.upload()  # выберите train.jsonl и val.jsonl

rows = [json.loads(line) for line in TRAIN.open(encoding='utf-8')]
print('train:', len(rows), '| val:', sum(1 for _ in VAL.open(encoding='utf-8')))
print('этапы:', sorted({r['stage'] for r in rows}))
print(rows[0]['conversations'][0]['content'][:200])


## 3. Модель


In [ ]:
from unsloth import FastModel
import torch

MAX_SEQ = 2048  # длиннее не нужно: медиана примера ~600 символов

model, tokenizer = FastModel.from_pretrained(
    model_name = 'unsloth/gemma-4-E2B-it',
    dtype = None,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
    full_finetuning = False,
)


In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 20260921,
)


## 4. Шаблон разговора

`gemma-4` — нетинкинговый шаблон, его же Unsloth советует для маленьких моделей.
Тот же шаблон обязан стоять при инференсе, иначе поведение в Ollama разойдётся
с ноутбуком.

Роли `system` у Gemma нет: шаблон требует строгого чередования user/assistant.
Системный промпт пайплайна приклеивается к первой реплике пользователя — тем же
способом и при обучении, и при проверке ниже.

Следствие для прода: пайплайн шлёт system и user двумя сообщениями, склеивать их
будет шаблон Ollama. После `ollama create` сделайте живой вызов и сравните ответ
с ноутбуком; если разойдётся — пропишите в Modelfile свой `TEMPLATE` с такой же
склейкой.


In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template = 'gemma-4')

def merge_system(convo):
    """Все ведущие system-реплики приклеиваем к первой реплике пользователя.

    Шаблон gemma-4 требует строгого чередования user/assistant и падает
    на роли system (а в части примеров их даже две подряд).
    """
    messages = [dict(m) for m in convo]
    head = []
    while messages and messages[0]['role'] == 'system':
        head.append(messages.pop(0)['content'])
    if head:
        if not messages:
            raise ValueError('разговор из одних system-реплик')
        messages[0]['content'] = '\n\n'.join(head + [messages[0]['content']])
    return messages


In [ ]:
def to_text(batch):
    texts = []
    for convo in batch['conversations']:
        text = tokenizer.apply_chat_template(
            merge_system(convo), tokenize = False, add_generation_prompt = False,
        )
        texts.append(text.removeprefix('<bos>'))  # <bos> добавит процессор
    return {'text': texts}

data = load_dataset('json', data_files = {'train': 'train.jsonl', 'val': 'val.jsonl'})
data = data.map(to_text, batched = True)
print(data['train'][0]['text'][:600])


## 5. Обучение


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = data['train'],
    eval_dataset = data['val'],
    args = SFTConfig(
        dataset_text_field = 'text',
        max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,  # 738 примеров — за одну эпоху формат не закрепится
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = 'epoch',
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 20260921,
        output_dir = 'outputs',
        report_to = 'none',
    ),
)


Считаем ошибку только на ответах модели: на системных промптах и чужих текстах учить
нечего, они в каждом примере почти одинаковые.


In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|turn>user\n',
    response_part = '<|turn>model\n',
)

# Проверка маскирования: должен остаться только ответ ассистента.
labels = trainer.train_dataset[0]['labels']
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in labels])
      .replace(tokenizer.pad_token, ' ').strip()[:400])


In [ ]:
stats = trainer.train()
print(stats.metrics)


## 6. Проверка до экспорта

Главное, что должно вырасти: ответ парсится как JSON, цитата дословная, лишних чисел нет.
Это те же проверки, что стоят в пайплайне, — гонять их надо до квантования, чтобы
отличить брак обучения от брака экспорта.

Отдельно смотрим температуру: Google рекомендует для Gemma 4 `temperature=1.0`, а пайплайн
шлёт `0.0` почти везде. Сравниваем оба режима на одних и тех же примерах.


In [ ]:
import json, re

val = [json.loads(l) for l in open('val.jsonl', encoding='utf-8')]

def answer(messages, temperature):
    text = tokenizer.apply_chat_template(
        merge_system(messages), tokenize = False, add_generation_prompt = True,
    )
    # Именованный text=: у Gemma 4 это процессор, и первый позиционный
    # аргумент у него images, а не текст.
    inputs = tokenizer(text = text, return_tensors = 'pt').to('cuda')
    out = model.generate(
        **inputs, max_new_tokens = 512, do_sample = temperature > 0,
        temperature = temperature or None, top_p = 0.95, top_k = 64,
    )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens = True)

def score(temperature, limit = 20):
    ok_json = ok_quote = total = 0
    for row in val[:limit]:
        convo = row['conversations']
        gold = convo[-1]['content']
        got = answer(convo[:-1], temperature)
        total += 1
        if not gold.lstrip().startswith('{'):
            continue
        match = re.search(r'\{.*\}', got, flags = re.DOTALL)
        if not match:
            continue
        try:
            payload = json.loads(match.group(0))
        except ValueError:
            continue
        ok_json += 1
        source = convo[-2]['content']
        quotes = re.findall(r'"quote":\s*"([^"]+)"', json.dumps(payload, ensure_ascii = False))
        if all(q in source for q in quotes):
            ok_quote += 1
    return ok_json, ok_quote, total

for t in (0.0, 1.0):
    print('temperature', t, '→ json/цитаты/всего:', score(t))


## 7. Экспорт в GGUF и Modelfile для Ollama

q4_k_m — тот же квант, что у остальных локальных моделей в проекте.


In [ ]:
model.save_pretrained_gguf('gemma4-e2b-fuckhr', tokenizer, quantization_method = 'q4_k_m')
!ls -lh gemma4-e2b-fuckhr


In [ ]:
MODELFILE = '''FROM ./unsloth.Q4_K_M.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0
PARAMETER top_p 0.95
PARAMETER top_k 64
'''
open('gemma4-e2b-fuckhr/Modelfile', 'w', encoding='utf-8').write(MODELFILE)
!cd gemma4-e2b-fuckhr && zip -r ../gemma4-e2b-fuckhr.zip . -x '*.bin'
from google.colab import files
files.download('gemma4-e2b-fuckhr.zip')


## 8. Что сделать на своей машине

```powershell
cd gemma4-e2b-fuckhr
ollama create fuckhr-e2b -f Modelfile
ollama list
```

Затем в `.env` одно имя на все профили — специализации по этапам больше нет:

```bash
LLM_LOCAL_MODEL_FAST=fuckhr-e2b
LLM_LOCAL_MODEL_SMART=fuckhr-e2b
LLM_LOCAL_MODEL_LONG=fuckhr-e2b
LLM_LOCAL_MODEL_LOCAL=fuckhr-e2b
LLM_LOCAL_MODEL_EMBEDDINGS=bge-m3
```

`LLM_LOCAL_STAGE_MODEL_*` оставьте пустыми: одна модель на все этапы — это и есть смысл
файнтюна. Раз модель одна и лёгкая, можно поднять `OLLAMA_MAX_LOADED_MODELS=2`, чтобы
эмбеддер не выгружался.

Проверка — бенчем на локальном маршруте, до и после:

```powershell
.venv\Scripts\python bench.py --models fuckhr-e2b --route local
```

или на странице «Модель» → «Сравнение моделей» → «Куда гонять: локальный адрес».
Сравнивать честно можно только с `qwen3:4b`/`qwen3:8b` на тех же кейсах: у бенча с
обучающей выборкой пересечений нет.
